# Taller de Algoritmos de Ordenamiento
**Universidad Industrial de Santander — Ingeniería en Ciencia de Datos**
**Estructuras de Datos**

Contenido del notebook:

1. Implementación de los algoritmos: inserción, burbuja, selección, Merge Sort y Quicksort.
2. Generación de un archivo de texto plano con 500.000 enteros, toma de muestras y medición de tiempos (5 repeticiones por combinación algoritmo–muestra, promediadas).
3. Gráficas comparativas, ajuste de la función que mejor describe cada algoritmo y análisis.

> **Nota sobre el enunciado:** la muestra "Cuentas mil números" se interpreta como **cincuenta mil (50.000)**, que es el valor que encaja entre 30.000 y 100.000.

> **Nota sobre el tiempo de ejecución:** los algoritmos cuadráticos (inserción, burbuja, selección) escritos en Python puro tardarían **muchas horas por corrida** con 500.000 datos (del orden de 10¹¹ operaciones elementales). Para que el taller se pueda ejecutar completo en Colab, cada algoritmo se escribe **una sola vez** en Python y se compila con **Numba** (`@njit`). El código del algoritmo es exactamente el mismo; Numba solo lo traduce a código máquina. Si se desea medir en Python puro, basta con poner `USAR_NUMBA = False` (en ese caso los algoritmos cuadráticos se limitan a muestras pequeñas; ver la celda de configuración).

## 0. Configuración

In [ ]:
# ===================== CONFIGURACIÓN =====================
USAR_NUMBA = True          # True: algoritmos compilados con Numba (recomendado)
REPETICIONES = 5           # repeticiones por combinación algoritmo–muestra
SEMILLA = 2026             # para que los resultados sean reproducibles
N_TOTAL = 500_000          # cantidad de enteros del archivo
RANGO_VALORES = (0, 1_000_000)   # rango de los enteros generados [min, max)
ARCHIVO = "numeros_500000.txt"

TAMANOS = [500, 1_000, 10_000, 30_000, 50_000,
           100_000, 200_000, 300_000, 400_000, 500_000]

# Solo aplica si USAR_NUMBA = False: tamaño máximo que se mide con los
# algoritmos cuadráticos en Python puro (más allá tardaría horas).
MAX_N_CUADRATICOS_PURO = 10_000
# =========================================================

In [ ]:
import time, os, math, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from numba import njit
except ImportError:
    !pip -q install numba
    from numba import njit

sys.setrecursionlimit(10_000)
print("NumPy", np.__version__)

## 1. Implementación de los algoritmos

Todos los algoritmos ordenan **in situ** un arreglo `a` (de NumPy) en orden ascendente.

### 1.1 Ordenamiento por inserción
Recorre el arreglo de izquierda a derecha; cada elemento se "inserta" en la posición correcta dentro de la parte ya ordenada, desplazando a la derecha los mayores que él.
- Peor caso / caso promedio: **O(n²)** — Mejor caso (ya ordenado): **O(n)**

In [ ]:
def insercion(a):
    n = len(a)
    for i in range(1, n):
        clave = a[i]
        j = i - 1
        while j >= 0 and a[j] > clave:
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = clave
    return a

### 1.2 Ordenamiento por burbuja
Compara pares adyacentes y los intercambia si están en desorden; en cada pasada el mayor elemento restante "sube" al final. Se incluye la optimización de detenerse si en una pasada no hubo intercambios.
- Peor caso / caso promedio: **O(n²)** — Mejor caso: **O(n)**

In [ ]:
def burbuja(a):
    n = len(a)
    for i in range(n - 1):
        hubo_intercambio = False
        for j in range(n - 1 - i):
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
                hubo_intercambio = True
        if not hubo_intercambio:
            break
    return a

### 1.3 Ordenamiento por selección
En cada pasada busca el mínimo de la parte no ordenada y lo coloca al inicio de esa parte.
- Siempre **O(n²)** comparaciones (no depende del orden inicial), pero solo O(n) intercambios.

In [ ]:
def seleccion(a):
    n = len(a)
    for i in range(n - 1):
        idx_min = i
        for j in range(i + 1, n):
            if a[j] < a[idx_min]:
                idx_min = j
        if idx_min != i:
            a[i], a[idx_min] = a[idx_min], a[i]
    return a

### 1.4 Ordenamiento por mezcla (Merge Sort)
Divide y vencerás: parte el arreglo en dos mitades, ordena cada una recursivamente y luego las **mezcla** en una sola secuencia ordenada (usando un arreglo auxiliar).
- Siempre **O(n log n)**; usa O(n) memoria adicional.

In [ ]:
def _mezclar(a, aux, ini, medio, fin):
    # mezcla a[ini:medio] y a[medio:fin], ambas ya ordenadas
    for k in range(ini, fin):
        aux[k] = a[k]
    i, j = ini, medio
    for k in range(ini, fin):
        if i >= medio:
            a[k] = aux[j]; j += 1
        elif j >= fin:
            a[k] = aux[i]; i += 1
        elif aux[j] < aux[i]:
            a[k] = aux[j]; j += 1
        else:
            a[k] = aux[i]; i += 1

def _merge_rec(a, aux, ini, fin):
    if fin - ini <= 1:
        return
    medio = (ini + fin) // 2
    _merge_rec(a, aux, ini, medio)
    _merge_rec(a, aux, medio, fin)
    _mezclar(a, aux, ini, medio, fin)

def merge_sort(a):
    aux = a.copy()
    _merge_rec(a, aux, 0, len(a))
    return a

### 1.5 Quicksort
Divide y vencerás: escoge un **pivote** (aquí, la mediana de tres: primero, medio y último), particiona el arreglo en menores y mayores que el pivote (esquema de Hoare) y ordena recursivamente cada parte. Para limitar la profundidad de la recursión se llama recursivamente sobre la parte más pequeña y se itera sobre la más grande.
- Caso promedio: **O(n log n)** — Peor caso: **O(n²)** (muy improbable con mediana de tres sobre datos aleatorios).

In [ ]:
def _particion(a, ini, fin):
    # fin es inclusivo. Pivote = mediana de tres
    medio = (ini + fin) // 2
    if a[medio] < a[ini]:
        a[medio], a[ini] = a[ini], a[medio]
    if a[fin] < a[ini]:
        a[fin], a[ini] = a[ini], a[fin]
    if a[fin] < a[medio]:
        a[fin], a[medio] = a[medio], a[fin]
    pivote = a[medio]
    i = ini - 1
    j = fin + 1
    while True:
        i += 1
        while a[i] < pivote:
            i += 1
        j -= 1
        while a[j] > pivote:
            j -= 1
        if i >= j:
            return j
        a[i], a[j] = a[j], a[i]

def _quick_rec(a, ini, fin):
    while ini < fin:
        p = _particion(a, ini, fin)
        # recursión sobre la parte pequeña, iteración sobre la grande
        if p - ini < fin - p:
            _quick_rec(a, ini, p)
            ini = p + 1
        else:
            _quick_rec(a, p + 1, fin)
            fin = p

def quicksort(a):
    _quick_rec(a, 0, len(a) - 1)
    return a

### 1.6 Versiones compiladas (Numba) y verificación
Si `USAR_NUMBA = True`, las mismas funciones se compilan con `njit`. Luego se verifica que cada algoritmo ordena correctamente comparando contra `np.sort`.

In [ ]:
if USAR_NUMBA:
    # Se compilan las funciones auxiliares primero para que las principales las usen
    _mezclar   = njit(cache=False)(_mezclar)
    _merge_rec = njit(cache=False)(_merge_rec)
    _particion = njit(cache=False)(_particion)
    _quick_rec = njit(cache=False)(_quick_rec)
    insercion  = njit(insercion)
    burbuja    = njit(burbuja)
    seleccion  = njit(seleccion)
    merge_sort = njit(merge_sort)
    quicksort  = njit(quicksort)

ALGORITMOS = {
    "Inserción":  insercion,
    "Burbuja":    burbuja,
    "Selección":  seleccion,
    "Merge Sort": merge_sort,
    "Quicksort":  quicksort,
}
CUADRATICOS = {"Inserción", "Burbuja", "Selección"}

# Verificación (la primera llamada además compila con Numba, así la
# compilación no se cuenta dentro de los tiempos medidos)
rng_test = np.random.default_rng(0)
casos = [np.array([], dtype=np.int64), np.array([5], dtype=np.int64),
         rng_test.integers(0, 10, 200), rng_test.integers(0, 10**6, 2000),
         np.arange(300, dtype=np.int64), np.arange(300, 0, -1).astype(np.int64)]
for nombre, f in ALGORITMOS.items():
    for c in casos:
        x = c.astype(np.int64).copy()
        f(x)
        assert np.array_equal(x, np.sort(c)), f"{nombre} falló"
    print(f"✔ {nombre} ordena correctamente")

## 2. Generación del archivo y toma de muestras

Se generan 500.000 enteros aleatorios (distribución uniforme) y se guardan en un archivo de texto plano, **un número por línea**. Luego se lee el archivo y de ese vector se toman las muestras (muestreo aleatorio sin reemplazo).

In [ ]:
rng = np.random.default_rng(SEMILLA)
numeros = rng.integers(RANGO_VALORES[0], RANGO_VALORES[1], size=N_TOTAL, dtype=np.int64)
np.savetxt(ARCHIVO, numeros, fmt="%d")
print(f"Archivo '{ARCHIVO}' creado: {os.path.getsize(ARCHIVO)/1e6:.2f} MB")

# Lectura del archivo de texto plano
vector = np.loadtxt(ARCHIVO, dtype=np.int64)
print("Elementos leídos:", len(vector))
print("Primeros 10:", vector[:10])

In [ ]:
# Una muestra por tamaño. Todos los algoritmos y repeticiones ordenan
# la MISMA muestra (cada vez sobre una copia), para que la comparación sea justa.
muestras = {}
for n in TAMANOS:
    if n == len(vector):
        muestras[n] = vector.copy()
    else:
        idx = rng.choice(len(vector), size=n, replace=False)
        muestras[n] = vector[idx]
pd.DataFrame({"Tamaño de muestra": TAMANOS,
              "Mínimo": [muestras[n].min() for n in TAMANOS],
              "Máximo": [muestras[n].max() for n in TAMANOS]})

### 2.1 Medición de tiempos
Para cada combinación algoritmo–muestra se ordena una copia de la muestra 5 veces, midiendo cada vez con `time.perf_counter()`. Tras cada ordenamiento se comprueba que el resultado esté ordenado (esa comprobación no entra en el tiempo).

Los resultados se guardan a medida que avanzan en `tiempos_crudos.csv`; si Colab se desconecta, al volver a ejecutar la celda se retoma donde iba.

> ⏱ Con Numba, la celda completa toma aproximadamente **entre 1 y 2 horas** en Colab: casi todo el tiempo se va en burbuja (≈ 10–15 min por repetición sumando todas las muestras), luego selección e inserción. Merge Sort y Quicksort tardan segundos. Mantenga la pestaña abierta; si la sesión se cae, vuelva a ejecutar las celdas desde el inicio y esta celda continúa donde quedó (los archivos `.txt` y `.csv` se conservan mientras no se reinicie el entorno de ejecución).

In [ ]:
CSV_CRUDO = "tiempos_crudos.csv"
if os.path.exists(CSV_CRUDO):
    crudos = pd.read_csv(CSV_CRUDO)
else:
    crudos = pd.DataFrame(columns=["algoritmo", "n", "repeticion", "tiempo_s"])
hechos = set(zip(crudos["algoritmo"], crudos["n"], crudos["repeticion"]))

def esta_ordenado(x):
    return bool(np.all(x[:-1] <= x[1:])) if len(x) > 1 else True

inicio_total = time.perf_counter()
for n in TAMANOS:
    for nombre, f in ALGORITMOS.items():
        if not USAR_NUMBA and nombre in CUADRATICOS and n > MAX_N_CUADRATICOS_PURO:
            continue
        for r in range(1, REPETICIONES + 1):
            if (nombre, n, r) in hechos:
                continue
            x = muestras[n].copy()
            t0 = time.perf_counter()
            f(x)
            t = time.perf_counter() - t0
            assert esta_ordenado(x), f"{nombre} no ordenó la muestra n={n}"
            crudos.loc[len(crudos)] = [nombre, n, r, t]
            crudos.to_csv(CSV_CRUDO, index=False)
        ult = crudos[(crudos.algoritmo == nombre) & (crudos.n == n)].tiempo_s.mean()
        print(f"n={n:>7,}  {nombre:<11} promedio = {ult:10.4f} s   "
              f"(transcurrido {(time.perf_counter()-inicio_total)/60:5.1f} min)")
print("Listo.")

### 2.2 Tabla de tiempos

Tiempos de las 5 repeticiones (en segundos) para cada combinación:

In [ ]:
crudos = pd.read_csv("tiempos_crudos.csv")
crudos["n"] = crudos["n"].astype(int)
orden_alg = list(ALGORITMOS)
tabla_reps = (crudos.pivot_table(index=["algoritmo", "n"], columns="repeticion",
                                 values="tiempo_s")
                    .reindex(orden_alg, level=0))
tabla_reps.columns = [f"Rep. {c}" for c in tabla_reps.columns]
tabla_reps["Promedio"] = tabla_reps.mean(axis=1)
tabla_reps["Desv. est."] = tabla_reps.iloc[:, :REPETICIONES].std(axis=1)
pd.set_option("display.float_format", lambda v: f"{v:.6f}")
tabla_reps

Tabla resumen de **tiempos promedio (s)**: filas = tamaño de muestra, columnas = algoritmo.

In [ ]:
promedios = (crudos.groupby(["n", "algoritmo"])["tiempo_s"].mean()
                   .unstack()[orden_alg])
promedios.index.name = "Tamaño de muestra"
promedios.to_csv("tiempos_promedio.csv")
with pd.ExcelWriter("tiempos_ordenamiento.xlsx") as w:
    tabla_reps.to_excel(w, sheet_name="Repeticiones")
    promedios.to_excel(w, sheet_name="Promedios")
promedios

## 3. Gráficas y ajuste de funciones

In [ ]:
colores = {"Inserción": "#1f77b4", "Burbuja": "#d62728", "Selección": "#ff7f0e",
           "Merge Sort": "#2ca02c", "Quicksort": "#9467bd"}
marcas = {"Inserción": "o", "Burbuja": "s", "Selección": "^",
          "Merge Sort": "D", "Quicksort": "v"}

fig, ejes = plt.subplots(1, 2, figsize=(15, 5.5))
for nombre in orden_alg:
    s = promedios[nombre].dropna()
    for ax in ejes:
        ax.plot(s.index, s.values, marker=marcas[nombre], color=colores[nombre],
                label=nombre, linewidth=1.8)
ejes[0].set_title("Tiempo promedio vs. tamaño (escala lineal)")
ejes[1].set_title("Tiempo promedio vs. tamaño (escala log–log)")
ejes[1].set_xscale("log"); ejes[1].set_yscale("log")
for ax in ejes:
    ax.set_xlabel("Tamaño de la muestra (n)")
    ax.set_ylabel("Tiempo promedio (s)")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
ejes[0].ticklabel_format(axis="x", style="plain")
plt.tight_layout(); plt.savefig("comparacion_algoritmos.png", dpi=150); plt.show()

En escala lineal los algoritmos O(n log n) quedan pegados al eje horizontal frente a los cuadráticos, por eso se grafican también por separado:

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(15, 5))
for ax, grupo, titulo in [(ejes[0], ["Inserción", "Burbuja", "Selección"], "Algoritmos cuadráticos"),
                          (ejes[1], ["Merge Sort", "Quicksort"], "Algoritmos divide y vencerás")]:
    for nombre in grupo:
        s = promedios[nombre].dropna()
        ax.plot(s.index, s.values, marker=marcas[nombre], color=colores[nombre], label=nombre)
    ax.set_title(titulo); ax.set_xlabel("Tamaño de la muestra (n)")
    ax.set_ylabel("Tiempo promedio (s)"); ax.grid(alpha=0.3); ax.legend()
    ax.ticklabel_format(axis="x", style="plain")
plt.tight_layout(); plt.savefig("comparacion_por_grupo.png", dpi=150); plt.show()

### 3.1 Ajuste de funciones

Para cada algoritmo se ajustan por **mínimos cuadrados** cuatro modelos candidatos de la forma $t(n) = a\,f(n) + b$:

| Modelo | $f(n)$ |
|---|---|
| Lineal | $n$ |
| Linearítmico | $n \log_2 n$ |
| Cuadrático | $n^2$ |
| Cúbico | $n^3$ |

y se elige el de mayor **R²**.

**Advertencia importante:** en el rango medido, $\log_2 n$ solo varía entre ≈ 9 y ≈ 19, así que para Merge Sort y Quicksort los modelos $a\,n+b$ y $a\,n\log_2 n+b$ dan R² prácticamente iguales (difieren en el tercer o cuarto decimal) y cualquiera de los dos puede quedar "ganando" por ruido de medición. Lo que sí se distingue sin ambigüedad es que **no** son cuadráticos. Por la teoría, la función que corresponde es $n\log n$.

Como verificación independiente se estima también el exponente $k$ en $t \approx c\,n^k$ mediante la pendiente de la recta en escala log–log (usando las muestras de 10.000 en adelante, donde el ruido de medición es despreciable): $k \approx 2$ indica comportamiento cuadrático y $k$ ligeramente mayor que 1 indica $n\log n$.

In [ ]:
MODELOS = {
    "a·n + b":        lambda n: n,
    "a·n·log2(n) + b": lambda n: n * np.log2(n),
    "a·n² + b":       lambda n: n**2,
    "a·n³ + b":       lambda n: n**3,
}

def ajustar(n, t, f):
    X = np.column_stack([f(n), np.ones_like(n)])
    (a, b), *_ = np.linalg.lstsq(X, t, rcond=None)
    pred = X @ np.array([a, b])
    r2 = 1 - np.sum((t - pred)**2) / np.sum((t - t.mean())**2)
    return a, b, r2

filas, mejores = [], {}
for nombre in orden_alg:
    s = promedios[nombre].dropna()
    n = s.index.values.astype(float); t = s.values
    res = {m: ajustar(n, t, f) for m, f in MODELOS.items()}
    mejor = max(res, key=lambda m: res[m][2])
    mejores[nombre] = (mejor, *res[mejor])
    grandes = n >= 10_000
    k = np.polyfit(np.log(n[grandes]), np.log(t[grandes]), 1)[0] if grandes.sum() >= 2 else np.nan
    fila = {"Algoritmo": nombre}
    fila.update({f"R² {m}": res[m][2] for m in MODELOS})
    fila["|ΔR²| n vs n·log n"] = abs(res["a·n + b"][2] - res["a·n·log2(n) + b"][2])
    fila.update({"Mejor modelo": mejor, "a": res[mejor][0], "b": res[mejor][1],
                 "Exponente k (log–log)": k})
    filas.append(fila)

ajustes = pd.DataFrame(filas).set_index("Algoritmo")
pd.set_option("display.float_format", lambda v: f"{v:.6g}")
ajustes

In [ ]:
fig, ejes = plt.subplots(2, 3, figsize=(17, 9))
ejes = ejes.ravel()
for ax, nombre in zip(ejes, orden_alg):
    s = promedios[nombre].dropna()
    n = s.index.values.astype(float); t = s.values
    modelo, a, b, r2 = mejores[nombre]
    nn = np.linspace(n.min(), n.max(), 400)
    ax.scatter(n, t, color=colores[nombre], zorder=3, label="Tiempo promedio medido")
    ax.plot(nn, a * MODELOS[modelo](nn) + b, "k--",
            label=f"Ajuste: {modelo}\na = {a:.3e}, b = {b:.3e}\nR² = {r2:.5f}")
    ax.set_title(nombre); ax.set_xlabel("n"); ax.set_ylabel("Tiempo (s)")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
    ax.ticklabel_format(axis="x", style="plain")

# Último panel: tiempos normalizados para ver la forma de crecimiento
ax = ejes[5]
for nombre in orden_alg:
    s = promedios[nombre].dropna()
    n = s.index.values.astype(float)
    ax.plot(n, s.values / s.values[-1], marker=marcas[nombre], color=colores[nombre], label=nombre)
ax.plot(n, (n / n[-1])**2, "k:", label="n² (referencia)")
ax.plot(n, (n*np.log2(n)) / (n[-1]*np.log2(n[-1])), "k-.", label="n log n (referencia)")
ax.set_title("Crecimiento normalizado (t / t_máx)"); ax.set_xlabel("n")
ax.grid(alpha=0.3); ax.legend(fontsize=8); ax.ticklabel_format(axis="x", style="plain")
plt.tight_layout(); plt.savefig("ajustes_por_algoritmo.png", dpi=150); plt.show()

### 3.2 Resumen numérico para el análisis

In [ ]:
n_max = promedios.index.max()
t_max = promedios.loc[n_max]
print(f"Con n = {n_max:,} elementos:")
for nombre in orden_alg:
    print(f"  {nombre:<11} {t_max[nombre]:12.4f} s")
rapido = t_max.idxmin()
print(f"\nEl más rápido es {rapido}.")
for nombre in orden_alg:
    if nombre != rapido:
        print(f"  {nombre} es {t_max[nombre]/t_max[rapido]:,.1f} veces más lento que {rapido}")

# Efecto de duplicar n (de 200.000 a 400.000)
if {200_000, 400_000} <= set(promedios.index):
    print("\nFactor de crecimiento del tiempo al duplicar n (200.000 → 400.000):")
    for nombre in orden_alg:
        print(f"  {nombre:<11} ×{promedios.loc[400_000, nombre]/promedios.loc[200_000, nombre]:.2f}")
    print("  (teórico: ×4 para n², ≈×2,1 para n log n)")

### 3.3 ¿Qué información sobre los algoritmos da la gráfica?

*(Revise que las cifras de la celda anterior respalden cada afirmación; los valores exactos dependen de la máquina en la que se ejecute.)*

1. **Dos familias claramente separadas.** Inserción, burbuja y selección forman un grupo cuyas curvas crecen como una **parábola**: el mejor ajuste es $a\,n^2 + b$ con R² cercano a 1 y el exponente log–log es $k \approx 2$. Merge Sort y Quicksort crecen casi en línea recta (exponente $k \approx 1$): los modelos $a\,n\log_2 n + b$ y $a\,n + b$ se ajustan prácticamente igual de bien, porque el factor $\log n$ crece muy despacio y en este rango se confunde con una constante. Esto es coherente con la complejidad teórica **O(n log n)**; lo que la gráfica muestra sin ambigüedad es la diferencia entre **O(n²)** y un crecimiento casi lineal.

2. **Escalabilidad.** Al duplicar el tamaño de la muestra, el tiempo de los algoritmos cuadráticos se multiplica aproximadamente por 4, mientras que el de Merge Sort y Quicksort se multiplica por algo más de 2. Por eso, con muestras pequeñas (500 o 1.000 datos) todos son prácticamente instantáneos y la diferencia no se nota, pero con 500.000 datos los cuadráticos tardan minutos y los de divide y vencerás fracciones de segundo: la diferencia es de **varios miles de veces**.

3. **Dentro de los cuadráticos.** Aunque los tres son O(n²), la constante $a$ cambia: **burbuja** suele ser el más lento (hace muchísimos intercambios, ≈ n²/4 en promedio), **selección** hace ≈ n²/2 comparaciones pero solo n intercambios, e **inserción** suele ser el más rápido de los tres porque en promedio solo recorre la mitad de la parte ordenada (≈ n²/4 desplazamientos) y desplaza en vez de intercambiar.

4. **Dentro de los O(n log n).** **Quicksort** suele ser más rápido que Merge Sort: ordena in situ (sin copiar a un arreglo auxiliar) y su ciclo de partición es muy simple. Merge Sort, en cambio, tiene un tiempo más predecible (siempre n log n, sin peor caso cuadrático) pero usa memoria adicional O(n).

5. **Conclusión práctica.** Para volúmenes de datos grandes solo son viables los algoritmos O(n log n). Los cuadráticos únicamente tienen sentido para arreglos muy pequeños o casi ordenados (caso en que inserción es O(n)); de hecho, muchas implementaciones reales de ordenamiento (p. ej. Timsort, introsort) usan inserción para los subarreglos pequeños.

### Archivos generados

In [ ]:
for f in ["numeros_500000.txt", "tiempos_crudos.csv", "tiempos_promedio.csv",
          "tiempos_ordenamiento.xlsx", "comparacion_algoritmos.png",
          "comparacion_por_grupo.png", "ajustes_por_algoritmo.png"]:
    print(f, "✔" if os.path.exists(f) else "✘")
# Para descargarlos desde Colab:
# from google.colab import files; files.download("tiempos_ordenamiento.xlsx")